<a href="https://colab.research.google.com/github/7235SYXD/Real-Estate/blob/main/DSP_on_Real_Estate_(NB_2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — INSTALL LIBRARIES                                      ║
# ╚══════════════════════════════════════════════════════════════════╝

import subprocess, sys
for lib in ["kagglehub","catboost","shap","optuna",
            "vaderSentiment","yake","textstat"]:
    subprocess.run([sys.executable,"-m","pip","install",lib,"-q"])
print("All libraries installed.")


All libraries installed.


In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — GOOGLE DRIVE                                           ║
# ╚══════════════════════════════════════════════════════════════════╝

import os, shutil
from google.colab import drive, files

drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/RealEstate_TXNY"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

Mounted at /content/drive
Save directory: /content/drive/MyDrive/RealEstate_TXNY


In [3]:
# ╔════════════════════════════════════════════════════╗
# ║  CELL 3 — IMPORT ALL LIBRARIES                     ║
# ╚════════════════════════════════════════════════════╝

import os
import re
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import gc # Import the garbage collection module

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import math
import matplotlib.patches as mpatches

# Sklearn
from sklearn.model_selection import train_test_split, KFold, cross_val_predict, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline as SKPipeline
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, f1_score, classification_report,
    accuracy_score, precision_score, recall_score, roc_curve,
    precision_recall_curve, average_precision_score
)
from sklearn.ensemble import HistGradientBoostingRegressor

# Gradient boosting
!pip install catboost
from catboost import CatBoostRegressor, CatBoostClassifier
import xgboost as xgb

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Hyperparameter tuning
!pip install optuna
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Interpretability
import shap

# NLP
!pip install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
!pip install yake
import yake
!pip install textstat
import textstat

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.4f}".format)

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("=" * 60)
print("All libraries imported successfully.")
print(f"  pandas     : {pd.__version__}")
print(f"  numpy      : {np.__version__}")
print(f"  tensorflow : {tf.__version__}")
print(f"  sklearn    : OK")
print("=" * 60)


All libraries imported successfully.
  pandas     : 2.2.2
  numpy      : 2.0.2
  tensorflow : 2.20.0
  sklearn    : OK


In [4]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  NOTEBOOK 2: SETUP -restore state from notebook 1                ║
# ╚══════════════════════════════════════════════════════════════════╝

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "dill", "-q"])
import dill, json, os

CKPT_DIR  = f"{SAVE_DIR}/checkpoints"
CKPT_NAME = "checkpoint_1_to_2"

with open(f"{CKPT_DIR}/{CKPT_NAME}.pkl", "rb") as f:
    _state = dill.load(f)
globals().update(_state)

keras_json_path = f"{CKPT_DIR}/{CKPT_NAME}_keras.json"
_keras_paths = {}
if os.path.exists(keras_json_path):
    with open(keras_json_path, "r") as f:
        content = f.read().strip()
        if content:
            _keras_paths = json.loads(content)
        else:
            print(f"  Warning: {keras_json_path} is empty. No Keras models to restore.")
else:
    print(f"  Warning: {keras_json_path} not found. No Keras models to restore.")

for _name, _path in _keras_paths.items():
    globals()[_name] = keras.models.load_model(_path)

print("=" * 65)
print(f"CHECKPOINT RESTORED <- {CKPT_DIR}/{CKPT_NAME}.pkl")
print("=" * 65)
print(f"  Variables restored    : {len(_state)}")
print(f"  Keras models restored : {list(_keras_paths.keys()) or 'none'}")
if "df_combined" in globals():
    print(f"  df_combined shape      : {df_combined.shape}")
print(f"\nNotebook 1 state loaded.")

CHECKPOINT RESTORED <- /content/drive/MyDrive/RealEstate_TXNY/checkpoints/checkpoint_1_to_2.pkl
  Variables restored    : 194
  Keras models restored : none
  df_combined shape      : (119979, 49)

Notebook 1 state loaded.


In [5]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 17 — FEATURE MATRIX + LEAK-FREE 80/10/10 SPLIT             ║
# ╚══════════════════════════════════════════════════════════════════╝
"""
Builds the final feature matrix and splits into train/val/test.

Final feature list (28 features):
  Structural (6)  : bed, bath, house_size, acre_lot, log_house_size, bed_bath_ratio
  Location  (3)   : is_tx_listing, is_ny_listing, zip_numeric,
  NLP       (14)  : nlp_vader_compound_mean/std, nlp_vader_pos_mean/std,
                    nlp_vader_neg_mean/std, nlp_flesch_ease_mean/std,
                    nlp_flesch_kincaid_mean/std, nlp_luxury_score_mean/std,
                    nlp_description_word_count_mean/std
  Derived   (3)   : price_per_sqft, latitude, longitude
  Encodings (2)   : zip_median_price, city_median_price

Task A target (regression):
    y_reg = df_combined["log_price"]

Task B target (classification):
    y_clf = df_combined["status_binary"]  (1=sold, 0=for_sale, NaN=other)
    Filtered to rows where status ∈ {"for_sale", "sold"} only.
    Coverage check: if < 5% of rows have valid status, Task B is skipped.

80/10/10 split:
    Stage 1: train_test_split(test_size=0.10)         → 90% train+val, 10% test
    Stage 2: train_test_split(test_size=0.10/0.90)    → 80% train, 10% val

Leak-free target encoding:
    Function: add_leak_free_encoding(X, z_series, c_series)
    zip_med_tr  = training rows only → median(price) grouped by zip_numeric
    city_med_tr = training rows only → median(price) grouped by city
    Val/test rows are mapped using training medians; unseen zip/city → global median.

Arrays prepared:
    Xtr_raw, Xva_raw, Xts_raw : raw numpy arrays for tree/linear models
    Xtr_sc, Xva_sc, Xts_sc    : imputed + StandardScaler arrays for MLP
    yr_tr_sc, yr_va_sc        : normalised log_price for MLP training

Helper functions defined:
    norm_y(y)   : standardise log_price to zero mean, unit variance
    denorm_y(y) : inverse of norm_y — converts MLP predictions back to log_price scale
"""

print("=" * 65)
print("CELL 17 — FEATURE MATRIX + TRAIN/VAL/TEST SPLIT")
print("=" * 65)

# Feature list
structural_feat = [c for c in [bed_col, bath_col, sqft_col, "acre_lot",
                                "log_house_size", "bed_bath_ratio"]
                   if c and c in df_combined.columns]
location_feat   = [c for c in ["is_tx_listing", "is_ny_listing",
                                "zip_numeric", "latitude", "longitude"]
                   if c in df_combined.columns]
derived_feat    = [c for c in ["price_per_sqft"] if c in df_combined.columns]
nlp_feat        = [c for c in df_combined.columns if c.startswith("nlp_")]

# Placeholder columns for leak-free encodings
encoding_feat   = ["zip_median_price", "city_median_price"]

base_features   = structural_feat + location_feat + derived_feat + nlp_feat
all_features    = base_features + encoding_feat
all_features    = list(dict.fromkeys(all_features))  # deduplicate

print(f"\nFeature groups:")
print(f"  Structural ({len(structural_feat)}) : {structural_feat}")
print(f"  Location   ({len(location_feat)})  : {location_feat}")
print(f"  Derived    ({len(derived_feat)})   : {derived_feat}")
print(f"  NLP        ({len(nlp_feat)})       : {nlp_feat[:4]} ...")
print(f"  Encodings  ({len(encoding_feat)})  : {encoding_feat}")
print(f"  TOTAL features: {len(all_features)}")

# Task A — regression
y_reg = df_combined["log_price"].copy()

# Task B classification — normalise status and filter to for_sale / sold
y_clf = None
if "status" in df_combined.columns:
    status_norm = df_combined["status"].astype(str).str.lower().str.strip()
    valid_mask  = status_norm.isin(["for_sale", "sold"])
    df_combined["status_binary"] = np.nan
    df_combined.loc[valid_mask, "status_binary"] = (
        status_norm[valid_mask] == "sold").astype(int)
    clf_coverage = valid_mask.mean()
    print(f"\nTask B status coverage (for_sale/sold): {clf_coverage:.1%}")
    if clf_coverage >= 0.05:
        y_clf = df_combined["status_binary"]
        print(f"  sold=1: {(y_clf==1).sum():,}  for_sale=0: {(y_clf==0).sum():,}")
    else:
        print("  Coverage <5% — Task B skipped")

# Price band for stratification
df_combined["price_band"] = pd.qcut(
    df_combined["price"], q=4, labels=["Q1","Q2","Q3","Q4"])

X_all    = df_combined[base_features].copy()
price_s  = df_combined["price"].copy()
city_s   = df_combined[city_col].copy() if city_col else pd.Series(["UNK"]*len(df_combined))
zip_s    = df_combined["zip_numeric"].copy() if "zip_numeric" in df_combined.columns \
           else pd.Series([np.nan]*len(df_combined))
pband_s  = df_combined["price_band"].astype(str)

X_trval, X_test, y_reg_trval, y_reg_test, \
p_trval, p_test, c_trval, c_test, \
z_trval, z_test, pb_trval, pb_test = train_test_split(
    X_all, y_reg, price_s, city_s, zip_s, pband_s,
    test_size=0.10, random_state=SEED, stratify=pband_s)

X_train, X_val, y_reg_train, y_reg_val, \
p_train, p_val, c_train, c_val, \
z_train, z_val = train_test_split(
    X_trval, y_reg_trval, p_trval, c_trval, z_trval,
    test_size=0.10/0.90, random_state=SEED, stratify=pb_trval)

print(f"\nSplit sizes:")
print(f"  Train : {len(X_train):,} rows ({len(X_train)/len(df_combined)*100:.0f}%)")
print(f"  Val   : {len(X_val):,} rows ({len(X_val)/len(df_combined)*100:.0f}%)")
print(f"  Test  : {len(X_test):,} rows ({len(X_test)/len(df_combined)*100:.0f}%)")

# Leak-free target encoding
print("\nComputing leak-free target encodings from training rows only")
g_global = float(p_train.median())

# zip_median_price
if "zip_numeric" in X_train.columns:
    zip_med_tr  = pd.concat([z_train, p_train], axis=1).groupby("zip_numeric")["price"].median()
    g_zip       = float(zip_med_tr.median())
else:
    zip_med_tr  = pd.Series(dtype=float)
    g_zip       = g_global

# city_median_price
cc = city_col if city_col else "city"
city_med_tr = pd.concat([c_train, p_train], axis=1).groupby(cc)["price"].median()
g_city      = float(city_med_tr.median())

def add_leak_free_encoding(X, z_series, c_series):
    """Add zip and city median price encodings using ONLY training medians."""
    X = X.copy()
    if "zip_numeric" in X.columns:
        X["zip_median_price"]  = z_series.map(zip_med_tr).fillna(g_zip)
    else:
        X["zip_median_price"]  = g_zip
    X["city_median_price"] = c_series.map(city_med_tr).fillna(g_city)
    return X

X_train = add_leak_free_encoding(X_train, z_train, c_train)
X_val   = add_leak_free_encoding(X_val,   z_val,   c_val)
X_test  = add_leak_free_encoding(X_test,  z_test,  c_test)

# Update final feature list
final_features = [c for c in all_features if c in X_train.columns]
final_features = list(dict.fromkeys(final_features))

print(f"Final feature count after encodings: {len(final_features)}")
print(f"Top correlations with log_price (train):")
tr_corr = X_train[final_features].corrwith(y_reg_train).abs().sort_values(ascending=False)
print(tr_corr.head(10).round(4).to_string())

# Classification split (Task B)
y_clf_train = y_clf_val = y_clf_test = None
if y_clf is not None:
    y_clf_train = y_clf.loc[X_train.index]
    y_clf_val   = y_clf.loc[X_val.index]
    y_clf_test  = y_clf.loc[X_test.index]
    clf_mask_tr  = y_clf_train.notna()
    clf_mask_val = y_clf_val.notna()
    clf_mask_tst = y_clf_test.notna()

# Raw numpy arrays for tree/linear models
Xtr_raw  = X_train[final_features].values
Xva_raw  = X_val[final_features].values
Xts_raw  = X_test[final_features].values

# Scaled arrays for MLP
mlp_pipe = SKPipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])
Xtr_sc   = mlp_pipe.fit_transform(Xtr_raw)
Xva_sc   = mlp_pipe.transform(Xva_raw)
Xts_sc   = mlp_pipe.transform(Xts_raw)
n_feat   = Xtr_sc.shape[1]

# MLP target normalisation
yr_tr_mean = float(y_reg_train.mean())
yr_tr_std  = float(y_reg_train.std())
def norm_y(y):   return (np.array(y) - yr_tr_mean) / yr_tr_std
def denorm_y(y): return np.array(y) * yr_tr_std + yr_tr_mean

yr_tr_sc = norm_y(y_reg_train.values)
yr_va_sc = norm_y(y_reg_val.values)

print("\nFeature matrix ready:")
print(f"  Tree/linear (raw)  : {Xtr_raw.shape}")
print(f"  MLP               : {Xtr_sc.shape}")
print(f"  NaN in Xtr_raw     : {np.isnan(Xtr_raw).sum():,}  "
      f"(zip_numeric 30% NaN — HGB handles natively)")
print(f"  NaN in Xtr_sc      : {np.isnan(Xtr_sc).sum()} ")

CELL 17 — FEATURE MATRIX + TRAIN/VAL/TEST SPLIT

Feature groups:
  Structural (6) : ['bed', 'bath', 'house_size', 'acre_lot', 'log_house_size', 'bed_bath_ratio']
  Location   (5)  : ['is_tx_listing', 'is_ny_listing', 'zip_numeric', 'latitude', 'longitude']
  Derived    (1)   : ['price_per_sqft']
  NLP        (14)       : ['nlp_vader_compound_mean', 'nlp_vader_compound_std', 'nlp_vader_pos_mean', 'nlp_vader_pos_std'] ...
  Encodings  (2)  : ['zip_median_price', 'city_median_price']
  TOTAL features: 28

Task B status coverage (for_sale/sold): 67.9%
  sold=1: 24,156  for_sale=0: 57,334

Split sizes:
  Train : 95,983 rows (80%)
  Val   : 11,998 rows (10%)
  Test  : 11,998 rows (10%)

Computing leak-free target encodings from training rows only
Final feature count after encodings: 28
Top correlations with log_price (train):
city_median_price      0.5483
zip_median_price       0.4712
price_per_sqft         0.4560
bath                   0.3511
bed                    0.2167
bed_bath_ratio    

In [6]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 18 — EVALUATION HELPER                                     ║
# ╚══════════════════════════════════════════════════════════════════╝
"""
Defines the shared evaluation function that is used throughout
the modelling pipeline.

Function defined:
    evaluate_reg(y_true, y_pred, label="", split="val") -> dict
        Computes regression metrics and prints a formatted summary line.

        Args:
            y_true (array-like) : actual log_price values
            y_pred (array-like) : predicted log_price values
            label  (str)        : model name for printing the output
            split  (str)        : "val" or "test" — shown in output

        Returns:
            dict with keys: model, split, R2, RMSE, MAE, Real_MAE_USD

        Metrics computed:
            RMSE        : sqrt(mean_squared_error(y_true, y_pred))
            MAE         : mean_absolute_error(y_true, y_pred)
            R2          : r2_score(y_true, y_pred)
            Real_MAE_USD: mean_absolute_error(expm1(y_true), expm1(y_pred))
                          Back-transforms from log scale to dollars.
                          Directly interpretable for real estate agents.

Lists initialised:
    results_baseline (list) : appended to by Cell 19A for each baseline model
    results_tuned    (list) : appended to by Cells 20–24 for each tuned model
    Both lists will be used in Cell 25 to build the comparison table and charts.
"""

def evaluate_reg(y_true, y_pred, label="", split="val"):
    #Evaluate regression predictions. Returns metric dict.
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    # Real-dollar MAE (back-transform from log scale)
    real_mae = mean_absolute_error(np.expm1(y_true), np.expm1(y_pred))
    print(f"  {label:42s} [{split}] "
          f"R²={r2:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}  "
          f"Real-MAE=${real_mae:,.0f}")
    return {"model": label, "split": split,
            "R2": r2, "RMSE": rmse, "MAE": mae, "Real_MAE_USD": real_mae}

results_baseline = []
results_tuned    = []

print("Evaluation helper defined")


Evaluation helper defined


In [7]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 19A — PRE-TUNING BASELINE (ALL 4 MODELS)                   ║
# ╚══════════════════════════════════════════════════════════════════╝
"""
Train all 4 models with default / standard parameters.
This is the benchmark that Optuna fine-tuning must beat.

Model 1 — Random Forest:
    n_estimators=100, n_jobs=-1 (all cores), random_state=SEED.
    Uses Xtr_rf (median-imputed raw arrays) — RF cannot handle NaN natively.
    The SimpleImputer fitted here (imp_rf) is then reused in Cell 25 for
    baseline test-set evaluation.

Model 2 — CatBoost:
    iterations=500, learning_rate=0.03, depth=6, verbose=0.
    Uses same Xtr_rf arrays as RF (CatBoost handles mixed types well
    when data is already numeric after imputation).
    eval_set passed to enable early stopping on validation set.

Model 3 — MLP:
    Architecture: Input → Dense(128) → BatchNorm → Dropout(0.3) → Dense(64) → Dense(1).
    Uses Xtr_sc (StandardScaler arrays) and yr_tr_sc (normalised target).
    EarlyStopping patience=5 — which is a very aggressive stop for baseline comparison.
    Predictions back-transformed via denorm_y() before evaluate_reg().

Model 4 — HistGradientBoosting:
    sklearn defaults (learning_rate=0.1, max_iter=100, max_depth=None).
    Uses Xtr_raw directly — HGB handles NaN natively so no imputation is needed.
    This is the key advantage of HGB over RF and CatBoost for this dataset
    where zip_numeric has ~30% NaN.

Output:
    results_baseline (list) : 4 dicts with R2, RMSE, MAE, Real_MAE_USD.
    baseline_df (pd.DataFrame) : formatted comparison table.
    Reports best baseline model by R² for reference.
"""

print("=" * 65)
print("CELL 19A — BASELINE RESULTS (ALL 4 MODELS)")
print("=" * 65)

# ── MODEL 1: Random Forest ─────────────────────────────────────────────────────
print("\n[1/4] Random Forest — default params")
from sklearn.impute import SimpleImputer
imp_rf = SimpleImputer(strategy="median")
Xtr_rf = imp_rf.fit_transform(Xtr_raw)
Xva_rf = imp_rf.transform(Xva_raw)
Xts_rf = imp_rf.transform(Xts_raw)

rf_base = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=SEED)
rf_base.fit(Xtr_rf, y_reg_train)
results_baseline.append(
    evaluate_reg(y_reg_val, rf_base.predict(Xva_rf), "RF (baseline)", "val"))

# ── MODEL 2: CatBoost ─────────────────────────────────────────────────────────
print("\n[2/4] CatBoost — default params")
cb_base = CatBoostRegressor(iterations=500, learning_rate=0.03,
                              depth=6, random_seed=SEED, verbose=0)
cb_base.fit(Xtr_rf, y_reg_train, eval_set=(Xva_rf, y_reg_val))
results_baseline.append(
    evaluate_reg(y_reg_val, cb_base.predict(Xva_rf), "CatBoost (baseline)", "val"))

# ── MODEL 3: MLP ──────────────────────────────────────────────────────────────
print("\n[3/4] MLP — default 2-layer [128, 64]")
from tensorflow.keras import layers as KL, Input as KInput, Model as KModel
mlp_inp = KInput(shape=(n_feat,))
mlp_x   = KL.Dense(128, activation="relu")(mlp_inp)
mlp_x   = KL.BatchNormalization()(mlp_x)
mlp_x   = KL.Dropout(0.3)(mlp_x)
mlp_x   = KL.Dense(64, activation="relu")(mlp_x)
mlp_x   = KL.Dense(1)(mlp_x)
mlp_base_model = KModel(inputs=mlp_inp, outputs=mlp_x)
mlp_base_model.compile(optimizer=keras.optimizers.Adam(0.001), loss="mse")
mlp_base_model.fit(Xtr_sc, yr_tr_sc,
                    validation_data=(Xva_sc, yr_va_sc),
                    epochs=50, batch_size=256, verbose=0,
                    callbacks=[keras.callbacks.EarlyStopping(
                        patience=5, restore_best_weights=True)])
mlp_base_preds = denorm_y(mlp_base_model.predict(Xva_sc, verbose=0).flatten())
results_baseline.append(
    evaluate_reg(y_reg_val, mlp_base_preds, "MLP (baseline)", "val"))

# ── MODEL 4: HistGradientBoosting ─────────────────────────────────────────────
print("\n[4/4] HistGradientBoosting — sklearn defaults")
from sklearn.ensemble import HistGradientBoostingRegressor
hgb_base = HistGradientBoostingRegressor(random_state=SEED)
# HGB handles NaN natively — pass raw arrays
hgb_base.fit(Xtr_raw, y_reg_train)
results_baseline.append(
    evaluate_reg(y_reg_val, hgb_base.predict(Xva_raw), "HGB (baseline)", "val"))

# ── Baseline comparison table ─────────────────────────────────────────────────
print(f"\n{'='*65}")
print("BASELINE SUMMARY TABLE (Validation Set)")
print("="*65)
baseline_df = pd.DataFrame(results_baseline)
print(baseline_df[["model","R2","RMSE","MAE","Real_MAE_USD"]].to_string(index=False))
print(f"\nBaseline complete   Best baseline R²: "
      f"{baseline_df['R2'].max():.4f} ({baseline_df.loc[baseline_df['R2'].idxmax(),'model']})")


CELL 19A — BASELINE RESULTS (ALL 4 MODELS)

[1/4] Random Forest — default params
  RF (baseline)                              [val] R²=0.7733  RMSE=0.4370  MAE=0.1642  Real-MAE=$55,454

[2/4] CatBoost — default params
  CatBoost (baseline)                        [val] R²=0.7548  RMSE=0.4545  MAE=0.2028  Real-MAE=$71,572

[3/4] MLP — default 2-layer [128, 64]
  MLP (baseline)                             [val] R²=0.7329  RMSE=0.4743  MAE=0.2432  Real-MAE=$81,201

[4/4] HistGradientBoosting — sklearn defaults
  HGB (baseline)                             [val] R²=0.7604  RMSE=0.4492  MAE=0.1992  Real-MAE=$70,321

BASELINE SUMMARY TABLE (Validation Set)
              model     R2   RMSE    MAE  Real_MAE_USD
      RF (baseline) 0.7733 0.4370 0.1642   55,453.8693
CatBoost (baseline) 0.7548 0.4545 0.2028   71,572.2726
     MLP (baseline) 0.7329 0.4743 0.2432   81,201.1952
     HGB (baseline) 0.7604 0.4492 0.1992   70,320.8870

Baseline complete   Best baseline R²: 0.7733 (RF (baseline))
